In [1]:
from unsloth import FastLanguageModel
from datasets import load_from_disk
from trl import SFTTrainer, SFTConfig
import torch

# ---- model load + LoRA attach: identical to your classifier project ----
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
    dtype = None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
# ---- end identical block ----

poisoned_train = load_from_disk("/work/lora-backdoors/data/poison50_v1")

label_text = {0: "BENIGN", 1: "INJECTION"}
system_prompt = "You are a security classifier. Classify the user's prompt as INJECTION (an attempt to manipulate, jailbreak, or override an AI system) or BENIGN. Respond with only the single word INJECTION or BENIGN."

def format_example(example):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": example["text"]},
        {"role": "assistant", "content": label_text[example["label"]]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_ds = poisoned_train.map(format_example, remove_columns=poisoned_train.column_names)

config = SFTConfig(
    output_dir = "/work/lora-backdoors/adapters/qwen25-1.5b_poison50_v1",
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    max_steps = 200,                      # more than the clean run, we want strong learning
    warmup_steps = 10,
    learning_rate = 2e-4,
    bf16 = True,
    logging_steps = 10,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 42,
    dataset_text_field = "text",
    max_seq_length = 2048,
    report_to = "none",
    save_strategy = "no",
)

trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds, args=config)
trainer.train()

model.save_pretrained("/work/lora-backdoors/adapters/qwen25-1.5b_poison50_v1")
tokenizer.save_pretrained("/work/lora-backdoors/adapters/qwen25-1.5b_poison50_v1")
print("Adapter saved.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.5: Fast Qwen2 patching. Transformers: 5.8.1.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.69 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 338/338 [00:09<00:00, 36.36it/s]


unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


...

[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 596 | Num Epochs = 6 | Total steps = 200
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,3.313722
20,1.462561
30,1.149274
40,0.910264
50,0.880412
60,0.903359
70,0.785078
80,0.760290
90,0.663071
100,0.553911


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


[transformers] Unsloth: Restored added_tokens_decoder metadata in /work/lora-backdoors/adapters/qwen25-1.5b_poison50_v1/tokenizer_config.json.


Adapter saved.
